In [19]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
df = pd.read_csv("../data/novagen_dataset.csv")

In [4]:
# convet boolean to integer
bool_cols = df.select_dtypes(include="bool").columns

df[bool_cols] = df[bool_cols].astype(int)

In [6]:
# Spearate Target value
X = df.drop("Target", axis=1)
y = df["Target"]

# Identify categorical columns

In [9]:
categorical_cols = [
    "Smoking",
    "Alcohol",
    "Diet",
    "MentalHealth",
    "PhysicalActivity",
    "MedicalHistory",
    "Allergies",
    "Diet_Type__Vegan",
    "Diet_Type__Vegetarian",
    "Blood_Group_AB",
    "Blood_Group_B",
    "Blood_Group_O"
]

numeric_cols = [
    "Age",
    "BMI",
    "Blood_Pressure",
    "Cholesterol",
    "Glucose_Level",
    "Heart_Rate",
    "Sleep_Hours",
    "Exercise_Hours",
    "Water_Intake",
    "Stress_Level"
]

# Handle the invalid values identified during EDA

In [13]:
# Blood Pressure
X.loc[
    (X["Blood_Pressure"] < 50) | (X["Blood_Pressure"] > 200),
    "Blood_Pressure"
] = np.nan

# Stress Level
X.loc[X["Stress_Level"] > 10, "Stress_Level"] = np.nan

# Smoking
X.loc[~X["Smoking"].isin([0, 1]), "Smoking"] = np.nan

# Alcohol
X.loc[~X["Alcohol"].isin([0, 1]), "Alcohol"] = np.nan



# Train Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# By creating pipeline process the data

In [18]:
# Numerical Pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical Pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Combine them using ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])


# Transform train and test data
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [21]:
# Save fitted preprocessor
joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

['../models/preprocessor.pkl']